In [1]:
import os
import json
from pathlib import Path
from typing import Any, Dict, List, TypedDict
from dotenv import load_dotenv

# --- SET YOUR PROJECT CONFIG HERE (Overrides .env) ---
os.environ["VERTEXAI_PROJECT"] = ""
os.environ["VERTEXAI_LOCATION"] = "us-central1"
os.environ["LLM_BACKEND"] = "vertex_ai"
os.environ["MODEL_NAME"] = "gemini-2.5-flash"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langgraph.graph import END, StateGraph

# Import local modules
from tools.docx_parser_tool import parse_docx
from tools.protocol_table_extractor import extract_tables
from tools.template_signal_detector import detect_update_units
from config.llm_config_langchain import get_llm

from state_schema import DemoState

# Setup base directory (Jupyter-friendly)
base_dir = Path(os.getcwd())
load_dotenv(base_dir / ".env")

print(f"LLM_BACKEND={os.getenv('LLM_BACKEND')}")
print(f"MODEL_NAME={os.getenv('MODEL_NAME')}")
print(f"VERTEXAI_PROJECT={os.getenv('VERTEXAI_PROJECT')}")

/home/jupyter/phds/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/phds/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/phds/lib/python3.10/site-packages/google/api_c

LLM_BACKEND=vertex_ai
MODEL_NAME=gemini-2.5-flash
VERTEXAI_PROJECT=


/home/jupyter/phds/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.vectorsearch_v1beta once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.vectorsearch_v1beta past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# ==========================================
# Helper Functions
# ==========================================
def _write_json(path: str, data) -> None:
    """Write JSON data to disk with UTF-8 encoding."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def _write_text(path: str, text: str) -> None:
    """Write plain text to disk."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(text, encoding="utf-8")


# ==========================================
# Node Definitions
# ==========================================
def parse_protocol_node(state: DemoState) -> DemoState:
    """Parse protocol.docx and save structured JSON."""
    parsed_protocol = parse_docx(state["protocol_path"])
    _write_json(state["protocol_json_path"], parsed_protocol)
    return {"parsed_protocol": parsed_protocol}


def parse_template_node(state: DemoState) -> DemoState:
    """Parse sap_template.docx and save structured JSON."""
    parsed_template = parse_docx(state["template_path"])
    _write_json(state["template_json_path"], parsed_template)
    return {"parsed_template": parsed_template}


def extract_tables_node(state: DemoState) -> DemoState:
    """Extract protocol tables from parsed protocol JSON."""
    protocol_tables = extract_tables(state["parsed_protocol"])
    _write_json(state["protocol_tables_json_path"], protocol_tables)
    return {"protocol_tables": protocol_tables}


def detect_signals_node(state: DemoState) -> DemoState:
    """Detect update units from the parsed SAP template."""
    update_units = detect_update_units(state["parsed_template"])
    _write_json(state["update_units_json_path"], update_units)
    return {"update_units": update_units}


def summarize_outputs_node(state: DemoState) -> DemoState:
    """
    Use the LLM only at the final step:
    - summarize what the workflow produced
    - lightly reformat the result into clean demo text
    """
    llm = get_llm()

    protocol_para_count = state["parsed_protocol"]["paragraph_count"]
    protocol_table_count = state["parsed_protocol"]["table_count"]
    template_para_count = state["parsed_template"]["paragraph_count"]
    template_table_count = state["parsed_template"]["table_count"]
    extracted_table_count = len(state["protocol_tables"])
    detected_unit_count = len(state["update_units"])

    prompt = f"""
You are helping explain a LangGraph demo for a short course on document automation in clinical trial documents.

Write a concise, clear summary in plain English for students.
Keep it short but informative. Use bullet points.

Facts:
- Protocol paragraphs: {protocol_para_count}
- Protocol tables found during parsing: {protocol_table_count}
- Protocol tables extracted into output artifact: {extracted_table_count}
- SAP template paragraphs: {template_para_count}
- SAP template tables: {template_table_count}
- Detected update units: {detected_unit_count}

Please explain:
1. What the workflow did
2. What files/artifacts were produced
3. Why this LangGraph version is more structured than pure Python
4. Why this version is more controlled than a CrewAI agent workflow
""".strip()

    response = llm.invoke(prompt)

    if hasattr(response, "content"):
        # Gemini models via VertexAI sometimes return a list of content blocks
        if isinstance(response.content, list):
            # Extract the actual text from the list of dictionaries
            summary_text = "\n".join([
                block.get("text", "") if isinstance(block, dict) else str(block)
                for block in response.content
            ])
        else:
            summary_text = response.content
    else:
        summary_text = str(response)

    _write_text(state["summary_txt_path"], summary_text)
    return {"summary_text": summary_text}




In [3]:
# ==========================================
# GRAPH BUILDING (Exercise Solution)
# ==========================================
def build_graph_exercise():
    """
    Build the LangGraph workflow with a Conditional Routing Edge.
    """
    graph = StateGraph(DemoState)

    # 1. Add all nodes to the graph
    graph.add_node("parse_protocol", parse_protocol_node)
    graph.add_node("parse_template", parse_template_node)
    graph.add_node("extract_tables", extract_tables_node)
    graph.add_node("detect_signals", detect_signals_node)
    graph.add_node("summarize_outputs", summarize_outputs_node)

    # 2. Set Entry Point and Linear Edges
    graph.set_entry_point("parse_protocol")
    graph.add_edge("parse_protocol", "parse_template")
    graph.add_edge("parse_template", "extract_tables")
    graph.add_edge("extract_tables", "detect_signals")

    # ---------------------------------------------------------
    # SOLUTION 1: The Routing Function
    # ---------------------------------------------------------
    def should_summarize(state: DemoState) -> Literal["summarize_outputs", "end"]:
        """Check if we have any update units. If not, skip the LLM summary."""
        units = state.get("update_units", [])
        
        if not units or len(units) == 0:
            print("\n>>> [ROUTER LOG] No update units detected. Skipping summary step. Fast Exit to END.")
            return "end"
        
        print(f"\n>>> [ROUTER LOG] Detected {len(units)} update units. Proceeding to LLM summary.")
        return "summarize_outputs"

    # ---------------------------------------------------------
    # SOLUTION 2: The Conditional Edge
    # ---------------------------------------------------------
    graph.add_conditional_edges(
        "detect_signals",   # The node that just finished
        should_summarize,   # The routing logic function
        {
            # If router returns "summarize_outputs", route to summarize_outputs node
            "summarize_outputs": "summarize_outputs", 
            # If router returns "end", route to the built-in END node
            "end": END 
        }
    )

    # 3. Final standard edge from summary to END
    graph.add_edge("summarize_outputs", END)

    return graph.compile()

In [4]:
import os
import json
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv


def main() -> None:
    # Handle Jupyter Notebook pathing or standard Python execution
    try:
        base_dir = Path(__file__).resolve().parent
    except NameError:
        base_dir = Path(os.getcwd())

    load_dotenv(base_dir / ".env")

    print(f"--- Running LangGraph Exercise ---")

    data_dir = base_dir / "data"
    output_dir = base_dir / "outputs"
    output_dir.mkdir(parents=True, exist_ok=True)

    initial_state = {
        "protocol_path": str(data_dir / "protocol.docx"),
        "template_path": str(data_dir / "sap_template.docx"),
        "protocol_json_path": str(output_dir / "protocol_parsed.json"),
        "template_json_path": str(output_dir / "sap_template_parsed.json"),
        "protocol_tables_json_path": str(output_dir / "protocol_tables.json"),
        "update_units_json_path": str(output_dir / "update_units.json"),
        "summary_txt_path": str(output_dir / "summary.txt"),
    }

    # BUILD THE EXERCISE GRAPH
    app = build_graph_exercise()
    
    # Optional: Print the graph structure to show students the conditional edge!
    print("Graph Structure Built:")
    try:
        print(app.get_graph().draw_ascii())
    except Exception:
        pass # Fallback if drawing is not supported in the environment

    # --- Configure graph execution limits ---
    run_config = {
        "recursion_limit": 50,
        "max_concurrency": 2
    }
    
    # Execute the graph
    final_state = app.invoke(initial_state, config=run_config)

    print("\n--- Execution Finished ---")
    if "summary_text" in final_state:
        print("Summary was generated because update_units were found!")
    else:
        print("Fast Exit Triggered! Summary was correctly skipped.")



In [5]:
if __name__ == "__main__":
    main()

--- Running LangGraph Exercise ---
Graph Structure Built:

>>> [ROUTER LOG] Detected 19 update units. Proceeding to LLM summary.


/home/jupyter/phdemo/ex3/config/llm_config_langchain.py:38: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  base_llm = ChatVertexAI(



--- Execution Finished ---
Summary was generated because update_units were found!
